# 02 · Yes/No probabilities on Colab

For each model: one forward pass per item, reading the next-token probabilities of "Yes" and "No" (summed over case and leading-space forms). No text is generated, so there is nothing to parse and each item takes a fraction of a second on a GPU.

1. **Runtime → Change runtime type → GPU** (T4 is enough; 7B models load in 4-bit).
2. Gated models (Llama, Gemma) need a Hugging Face token with access; the default list below uses ungated models only.
3. Run once with `SMOKE_TEST = True` (24 items per model) and look at the diagnostics, then set it to `False`.
4. Results go to Google Drive, so they survive a disconnect. Re-running resumes where it stopped. Copy the `logprobs-*.jsonl` files to `data/interim/` in the local repo for notebook 03.

In [ ]:
import os
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/laudima/causal-evaluation.git"

if IN_COLAB:
    ROOT = Path("/content/causal-evaluation")
    if not ROOT.exists():
        !git clone -q {REPO_URL} {ROOT}
    !pip install -q -e "{ROOT}[colab]"
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    OUT_DIR = Path("/content/drive/MyDrive/causal-evaluation/results")
else:
    OUT_DIR = Path(os.environ.get("CE_INTERIM", ROOT / "data" / "interim"))
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(OUT_DIR)

## Rebuild the sample and check it matches notebook 01

In [ ]:
from causal_evaluation.data.variants import build_variant_sample, write_jsonl

SEED = 20260903
SIZE_PER_VARIANT = 1000
EXPECTED_SHA = None  # paste "sample_sha256" from data/interim/variants-sample.manifest.json

items = build_variant_sample(ROOT / "data" / "raw", SIZE_PER_VARIANT, SEED)
sample_sha = write_jsonl(items, OUT_DIR / "variants-sample.jsonl")
print(f"{len(items):,} items, SHA-256 {sample_sha}")
if EXPECTED_SHA:
    assert sample_sha == EXPECTED_SHA, "Sample differs from the local one"

## Models

In [ ]:
# from huggingface_hub import login; login()  # only for gated models

# dtype: float32 keeps P(Yes) independent of batching; models that do not fit a T4 (15 GB)
# in float32 load in 4-bit with float16 compute. Report the dtype of each model in the paper.
MODELS = [
    {"name": "HuggingFaceTB/SmolLM2-1.7B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-1.5B-Instruct", "dtype": "float32", "load_in_4bit": False, "batch_size": 8},
    {"name": "microsoft/Phi-3.5-mini-instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
    {"name": "Qwen/Qwen2.5-7B-Instruct", "dtype": "float16", "load_in_4bit": True, "batch_size": 8},
]
SMOKE_TEST = True
SMOKE_ITEMS = 24


def result_path(model_name):
    suffix = "-smoke" if SMOKE_TEST else ""
    return OUT_DIR / f"logprobs-{model_name.split('/')[-1]}{suffix}.jsonl"

## Batch-invariance check (smoke test only)

P(Yes) for an item should not depend on which other items share its batch. Values above about 0.01 mean the precision setting for that model should change.

In [ ]:
from causal_evaluation.models.logprob import check_batch_invariance

if SMOKE_TEST:
    probe = [items[0], items[1500], items[2999], items[7]]
    for spec in MODELS:
        diff = check_batch_invariance(spec["name"], probe, spec["load_in_4bit"], spec["dtype"])
        print(f"{spec['name']}: max |batched - alone| = {diff:.4f}" + ("  <-- check dtype" if diff > 0.01 else ""))

## Run

In [ ]:
import gc

import torch

from causal_evaluation.models.logprob import run_model

run_items = items[:SMOKE_ITEMS] if SMOKE_TEST else items
for spec in MODELS:
    path = result_path(spec["name"])
    scored = run_model(
        run_items,
        spec["name"],
        path,
        batch_size=spec["batch_size"],
        load_in_4bit=spec["load_in_4bit"],
        dtype=spec["dtype"],
    )
    print(f"{spec['name']}: {scored} new items -> {path.name}")
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

## Diagnostics

`answer_mass` is the probability the model puts on Yes/No tokens at all. If its median is low (say below 0.5) the model wants to start with something else (look at `top_token`) and P(Yes) is less meaningful for that model.

In [ ]:
import pandas as pd

from causal_evaluation.evaluation.variant_analysis import load_scored, rate_table

paths = [p for p in sorted(OUT_DIR.glob("logprobs-*.jsonl")) if p.name.endswith("-smoke.jsonl") == SMOKE_TEST]
scored = load_scored(OUT_DIR / "variants-sample.jsonl", paths)
display(scored.groupby("model").agg(items=("item_id", "size"), median_answer_mass=("answer_mass", "median"), yes_rate=("pred", lambda s: (s == "yes").mean())))
display(scored.groupby("model")["top_token"].agg(lambda s: s.value_counts().head(3).to_dict()))
rate_table(scored, ["model", "variant"])